<a href="https://colab.research.google.com/github/WallofWonder/Hugging-Face-RL-Learning/blob/main/notebooks/unit2/unit2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第二单元：使用FrozenLake-v1 ⛄ 和 Taxi-v3 🚕 进行Q学习

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/thumbnail.jpg" alt="Unit 2 Thumbnail">

在这个笔记本中，**你将从零开始编写你的第一个强化学习智能体**，使用Q学习玩FrozenLake ❄️，并与社区分享，以及尝试不同的配置。

⬇️ 这是一个你**在几分钟内就能实现**的例子。⬇️

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="环境"/>

###🎮 环境：

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/)

###📚 RL-库：

- Python 和 NumPy
- [Gymnasium](https://gymnasium.farama.org/)

我们一直在努力改进教程，所以**如果你发现本笔记本中的一些问题**，请[在GitHub仓库上提出问题](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆

完成本笔记本后，你将能够：

- 使用**Gymnasium**，环境库。
- 从头开始编写一个Q学习智能体。
- **将你训练好的智能体和代码推送到Hub**，附带精美的视频回放和评估分数 🔥。

在这个免费课程中，你将：

- 📖 **理论和实践**学习深度强化学习。
- 🧑‍💻 学习使用**著名的深度强化学习库**，如Stable Baselines3、RL Baselines3 Zoo、CleanRL和Sample Factory 2.0。
- 🤖 在**独特环境中训练智能体**

更多信息请查看📚 课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">注册课程</a>**（我们正在收集你的电子邮件，以便**在每个单元发布时向你发送链接，并向你提供有关挑战和更新的信息）。**

保持联系的最佳方式是加入我们的Discord服务器，与社区和我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 先决条件 🏗️

在深入学习笔记本之前，你需要：

🔲 📚 **阅读单元2，学习Q学习** [Q-Learning by reading Unit 2](https://huggingface.co/deep-rl-course/unit2/introduction) 🤗

## Q学习小结

*Q学习* **是一种强化学习算法，它**：

- 训练*Q函数*，这是一种**动作值函数**，通过*Q表*在内部内存中编码，该**Q表包含所有状态-动作对的值。**
    
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Q function"  width="100%"/>

- 给定一个状态和动作，我们的Q函数**将在Q表中搜索相应的值。**
    
- 当训练完成时，**我们拥有一个最优的Q函数，因此也拥有一个最优的Q表。**
    
- 如果我们**有一个最优的Q函数**，我们就有了一个最优的策略，因为我们**知道对于每个状态，应该采取的最佳动作。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Link value policy"  width="100%"/>


但是，在一开始，我们的**Q表是无用的，因为它为每个状态-动作对赋予了任意值（大多数情况下我们将Q表初始化为0）**。但是，随着我们探索环境并更新Q表，它将为我们提供越来越好的近似值。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="q-learning.jpeg" width="100%"/>

这是Q学习的伪代码：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

# 让我们编写我们的第一个强化学习算法 🚀

要验证此动手练习是否符合[认证流程](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process)的要求，你需要将训练好的Taxi模型推送到Hub并**获得 >= 4.5 的结果**。

要查找你的结果，请访问[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)并找到你的模型，**结果 = 平均奖励 - 奖励标准差**

有关认证流程的更多信息，请查看此部分 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 安装依赖并创建虚拟显示 🔽

在此笔记本中，我们需要生成一个重播视频。为此，在Colab中，**我们需要一个虚拟屏幕来渲染环境**（从而录制帧）。

因此，以下单元格将安装库并创建并运行一个虚拟屏幕 🖥

我们将安装多个库：

- `gymnasium`：包含FrozenLake-v1 ⛄ 和 Taxi-v3 🚕 环境。
- `pygame`：用于FrozenLake-v1和Taxi-v3 UI。
- `numpy`：用于处理我们的Q表。

Hugging Face Hub 🤗 作为一个中心平台，任何人都可以共享和探索模型和数据集。它具有版本控制、度量、可视化和其他功能，可以让你轻松地与他人协作。

你可以在这里查看所有可用的深度强化学习模型（如果它们使用Q学习） 👉 https://huggingface.co/models?other=q-learning

In [ ]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit2/requirements-unit2.txt

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

为了确保使用新安装的库，**有时需要重新启动笔记本运行时**。下一个单元格将强制**运行时崩溃，因此你需要重新连接并从这里开始运行代码**。通过这个技巧，**我们将能够运行我们的虚拟屏幕。**

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 导入包 📦

除了已安装的库，我们还使用：

- `random`：用于生成随机数（对epsilon-greedy策略很有用）。
- `imageio`：用于生成重播视频。

In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle5 as pickle
from tqdm.notebook import tqdm

我们现在准备编写Q学习算法 🔥

# 第一部分：冰湖 ⛄ (非湿滑版本)

## 创建并理解[冰湖环境 ⛄]((https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
---

💡 开始使用环境时，一个好习惯是查看其文档

👉 https://gymnasium.farama.org/environments/toy_text/frozen_lake/

---

我们将训练我们的Q学习智能体**通过只在冰冻的瓷砖（F）上行走并避开洞（H）来从起始状态（S）导航到目标状态（G）**。

我们可以有两种大小的环境：

- `map_name="4x4"`：一个4x4的网格版本
- `map_name="8x8"`：一个8x8的网格版本


环境有两种模式：

- `is_slippery=False`：由于冰湖的非湿滑性质（确定性），智能体始终**按预期方向移动**。
- `is_slippery=True`：由于冰湖的湿滑性质（随机性），智能体**可能不总是按预期方向移动**。

现在让我们保持简单，使用4x4地图和非湿滑版本。
我们添加了一个名为`render_mode`的参数，它指定了环境应如何可视化。在我们的例子中，因为我们**希望在最后录制环境的视频，所以我们需要将`render_mode`设置为`rgb_array`**。

正如[文档中所解释的](https://gymnasium.farama.org/api/env/#gymnasium.Env.render)，“rgb_array”：返回一个代表环境当前状态的单帧。帧是一个`np.ndarray`，其形状为(x, y, 3)，表示x乘y像素图像的RGB值。

In [ ]:
# Create the FrozenLake-v1 environment using 4x4 map and non-slippery version and render_mode="rgb_array"
env = gym.make() # TODO use the correct parameters

### 解决方案

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

你可以像这样创建自己的自定义网格：

```python
desc=["SFFF", "FHFH", "FFFH", "HFFG"]
gym.make('FrozenLake-v1', desc=desc, is_slippery=True)
```

但我们现在将使用默认环境。

### 让我们看看环境是什么样子的：

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # Get a random observation

我们看到`Observation Space Shape Discrete(16)`表示观察是一个整数，代表**智能体的当前位置为`current_row * ncols + current_col`（其中行和列都从0开始）**。

例如，在4x4地图中，目标位置可以按如下方式计算：3 * 4 + 3 = 15。可能的观察数量取决于地图的大小。**例如，4x4地图有16种可能的观察。**


例如，state = 0 看起来像这样：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

动作空间（智能体可以采取的可能动作集）是离散的，有4个可用动作🎮：
- 0：向左移动
- 1：向下移动
- 2：向右移动
- 3：向上移动

奖励函数💰：
- 到达目标：+1
- 掉入洞中：0
- 到达冰面：0

## 创建并初始化Q表 🗄️

(👀 伪代码的步骤1)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


是时候初始化我们的Q表了！为了知道要使用多少行（状态）和列（动作），我们需要知道动作空间和观察空间。我们之前已经知道它们的值，但我们希望以编程方式获取它们，以便我们的算法适用于不同的环境。Gym提供了这样做的方法：`env.action_space.n`和`env.observation_space.n`

In [ ]:
state_space =
print("There are ", state_space, " possible states")

action_space =
print("There are ", action_space, " possible actions")

In [ ]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

### 解决方案

In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [ ]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

## 定义贪婪策略 🤖

请记住，我们有两种策略，因为Q学习是一种**离策略**算法。这意味着我们正在使用**不同的策略来行动和更新值函数**。

- Epsilon-greedy策略（行动策略）
- 贪婪策略（更新策略）

贪婪策略也将是Q学习智能体完成训练时我们拥有的最终策略。贪婪策略用于使用Q表选择动作。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Q-Learning" width="100%"/>

In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action =

  return action

#### 解决方案

In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action = np.argmax(Qtable[state][:])

  return action

## 定义epsilon-greedy策略 🤖

Epsilon-greedy是处理探索/利用权衡的训练策略。

epsilon-greedy的想法是：

- 以*概率1 - ɛ*：**我们进行利用**（即我们的智能体选择具有最高状态-动作对值的动作）。

- 以*概率ɛ*：我们进行**探索**（尝试一个随机动作）。

随着训练的继续，我们逐渐**减少epsilon值，因为我们需要越来越少的探索和越来越多的利用。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Q-Learning" width="100%"/>

In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num =
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action =
  # else --> exploration
  else:
    action = # Take a random action

  return action

#### 解决方案

In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num = random.uniform(0,1)
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action = greedy_policy(Qtable, state)
  # else --> exploration
  else:
    action = env.action_space.sample()

  return action

## 定义超参数 ⚙️

与探索相关的超参数是一些最重要的参数。

- 我们需要确保我们的智能体**充分探索状态空间**以学习一个好的值近似。为此，我们需要逐步衰减epsilon。
- 如果你衰减epsilon过快（衰减率过高），**你将面临智能体卡住的风险**，因为你的智能体没有充分探索状态空间，因此无法解决问题。

In [ ]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7          # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"     # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate
eval_seed = []               # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05            # Minimum exploration probability
decay_rate = 0.0005            # Exponential decay rate for exploration prob

## 创建训练循环方法

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

训练循环如下：

```
对于总训练集中的每个episode：

  减少epsilon（因为我们需要越来越少的探索）
  重置环境

  对于最大时间步中的每一步：
    使用epsilon-greedy策略选择动作At
    执行动作(a)并观察结果状态(s')和奖励(r)
    使用Bellman方程更新Q值Q(s,a) = Q(s,a) + lr * [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    如果完成，结束episode
    我们的下一个状态是新状态
```

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action =

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info =

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] =

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

#### 解决方案

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action = epsilon_greedy_policy(Qtable, state, epsilon)

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

## 训练Q学习智能体 🏃

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

## 让我们看看现在的Q学习表是什么样子 👀

In [ ]:
Qtable_frozenlake

## 评估方法 📝

- 我们定义了将用于测试Q学习智能体的评估方法。

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## 评估我们的Q学习智能体 📈

- 通常，你的平均奖励应为1.0。
- **环境相对简单**，因为状态空间非常小（16）。你可以尝试[将其替换为湿滑版本](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)，这将引入随机性，使环境更加复杂。

In [ ]:
# Evaluate our Agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

## 将我们训练的模型发布到Hub 🔥

现在我们看到了训练后的良好结果，**我们可以用一行代码将我们训练好的模型发布到Hub 🤗**。

这是一个模型卡的示例：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/modelcard.png" alt="Model card" width="100%"/>

在底层，Hub使用基于git的仓库（如果你不知道git是什么，不用担心），这意味着你可以在实验和改进智能体时用新版本更新模型。

#### Do not modify this code

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Generate a replay video of the agent
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
  """
  images = []
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not terminated or truncated:
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def push_to_hub(
    repo_id, model, env, video_fps=1, local_repo_path="hub"
):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat()
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  ```python

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  ```
  """

    evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

### .

通过使用`push_to_hub`，你**评估、录制回放、生成智能体的模型卡并将其推送到Hub**。

这样：
- 你可以**展示我们的工作** 🔥
- 你可以**可视化你的智能体玩耍** 👀
- 你可以**与社区分享一个其他人可以使用的智能体** 💾
- 你可以**访问排行榜 🏆 查看你的智能体与同学相比表现如何** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard

为了能够与社区分享你的模型，还需要遵循三个步骤：

1️⃣ (如果尚未完成) 创建一个HF帐户 ➡ https://huggingface.co/join

2️⃣ 登录，然后你需要从Hugging Face网站存储你的身份验证令牌。
- 创建一个新令牌 (https://huggingface.co/settings/tokens) **并授予写入角色**


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果你不想使用Google Colab或Jupyter Notebook，你需要改用此命令：`huggingface-cli login` (或 `login`)

3️⃣ 我们现在准备使用`push_to_hub()`函数将训练好的智能体推送到🤗 Hub 🔥

- 让我们创建**包含超参数和Q表的模型字典**。

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

让我们填写`push_to_hub`函数：

- `repo_id`：将要创建/更新的Hugging Face Hub存储库的名称
`(repo_id = {username}/{repo_name})`
💡 一个好的`repo_id`是`{username}/q-{env_id}`
- `model`：包含超参数和Qtable的模型字典。
- `env`：环境。
- `commit_message`：提交消息

In [ ]:
model

In [ ]:
username = "" # FILL THIS
repo_name = "q-FrozenLake-v1-4x4-noSlippery"
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

恭喜🥳 你刚刚从头开始实现、训练并上传了你的第一个强化学习智能体。
FrozenLake-v1 non-slippery是一个非常简单的环境，让我们尝试一个更难的环境 🔥。

# 第二部分：出租车-v3 🚖

## 创建并理解[出租车-v3 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)
---

💡 开始使用环境时，一个好习惯是查看其文档

👉 https://gymnasium.farama.org/environments/toy_text/taxi/

---

在`Taxi-v3` 🚕中，网格世界中有四个指定位置，分别用R（红色）、G（绿色）、Y（黄色）和B（蓝色）表示。

当episode开始时，**出租车从一个随机方块开始**，乘客在一个随机位置。出租车开到乘客的位置，**接载乘客**，开到乘客的目的地（四个指定位置中的另一个），然后**放下乘客**。一旦乘客放下，episode结束。


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

由于有25个出租车位置、5个可能的乘客位置（包括乘客在出租车内的情况）和4个目的地位置，因此有**500个离散状态**。

In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

In [ ]:
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

动作空间（智能体可以采取的可能动作集）是离散的，有**6个可用动作🎮**：

- 0：向南移动
- 1：向北移动
- 2：向东移动
- 3：向西移动
- 4：接载乘客
- 5：放下乘客

奖励函数💰：

- 每步-1，除非触发其他奖励。
- 成功送达乘客+20。
- 非法执行“接载”和“放下”动作-10。

In [ ]:
# Create our Q table with state_size rows and action_size columns (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Q-table shape: ", Qtable_taxi .shape)

## 定义超参数 ⚙️

⚠ 不要修改 EVAL_SEED：`eval_seed`数组**允许我们用相同的出租车起始位置评估你的智能体，以确保所有同学都获得相同的评估结果**

In [ ]:
# Training parameters
n_training_episodes = 25000   # Total training episodes
learning_rate = 0.7           # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# DO NOT MODIFY EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148] # Evaluation seed, this ensures that all classmates agents are trained on the same taxi starting position
                                                          # Each seed has a specific starting state

# Environment parameters
env_id = "Taxi-v3"           # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05           # Minimum exploration probability
decay_rate = 0.005            # Exponential decay rate for exploration prob


## 训练我们的Q学习智能体 🏃

In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

## 创建模型字典 💾 并将我们训练好的模型发布到Hub 🔥

- 我们创建一个模型字典，其中将包含所有训练超参数以实现可复现性以及Q表。

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

In [ ]:
username = "" # FILL THIS
repo_name = "" # FILL THIS
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

现在它已经在Hub上了，你可以使用排行榜🏆 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard 将你的Taxi-v3结果与你的同学进行比较。


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi-leaderboard.png" alt="Taxi Leaderboard">

# 第三部分：从Hub加载 🔽

Hugging Face Hub 🤗 的惊人之处在于你可以轻松地从社区加载强大的模型。

从Hub加载已保存的模型非常简单：

1. 访问 https://huggingface.co/models?other=q-learning 查看所有Q学习已保存模型的列表。
2. 选择一个并复制其repo_id

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/copy-id.png" alt="Copy id">

3. 然后我们只需使用`load_from_hub`，并提供：
- repo_id
- 文件名：仓库中保存的模型文件。

#### 不要修改此代码

In [ ]:
from urllib.error import HTTPError

from huggingface_hub import hf_hub_download


def load_from_hub(repo_id: str, filename: str) -> str:
    """
    Download a model from Hugging Face Hub.
    :param repo_id: id of the model repository from the Hugging Face Hub
    :param filename: name of the model zip file from the repository
    """
    # Get the model from the Hub, download and cache the model on your local disk
    pickle_model = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )

    with open(pickle_model, 'rb') as f:
      downloaded_model_file = pickle.load(f)

    return downloaded_model_file

### .

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-Taxi-v3", filename="q-learning.pkl") # Try to use another model

print(model)
env = gym.make(model["env_id"])

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-FrozenLake-v1-no-slippery", filename="q-learning.pkl") # Try to use another model

env = gym.make(model["env_id"], is_slippery=False)

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

## 一些额外的挑战 🏆

学习**最好的方式就是亲自尝试**！正如你所看到的，当前的智能体表现并不出色。第一个建议是，你可以训练更多步。在1,000,000步之后，我们看到了很好的结果！

在[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)上你会找到你的智能体。你能登上榜首吗？

以下是一些登上排行榜的建议：

* 训练更多步
* 通过查看同学们的做法来尝试不同的超参数。
* **将你新训练的模型推送到Hub** 🔥

在冰上行走和驾驶出租车对你来说太无聊了吗？尝试**改变环境**，为什么不使用FrozenLake-v1湿滑版本呢？[查阅gymnasium文档](https://gymnasium.farama.org/)了解它们是如何工作的，然后尽情玩乐吧🎉。

_____________________________________________________________________
恭喜🥳，你刚刚从头实现、训练并上传了你的第一个强化学习智能体。

理解Q学习是**理解基于值的方法的重要一步。**

在下一个单元，深度Q学习中，我们将看到虽然创建和更新Q表是一个很好的策略——**然而，它不可扩展。**

例如，想象你创建一个智能体来玩Doom游戏。

<img src="https://vizdoom.cs.put.edu.pl/user/pages/01.tutorial/basic.png" alt="Doom"/>

Doom是一个拥有巨大状态空间（数百万个不同状态）的大型环境。为该环境创建和更新Q表将是低效的。

这就是为什么我们将在下一个单元学习深度Q学习，这是一种**我们使用神经网络来近似给定状态下每个动作的Q值**的算法。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>

单元3见！🔥

## 继续学习，保持出色 🤗